### Bronze Layer — Online Retail II Ingestion

In [0]:
%pip install openpyxl

In [0]:
%restart_python

In [0]:
import pandas as pd
import re

file_path = "/Volumes/workspace/default/raw_data/online_retail_II.xlsx"

sheet_2009_2010 = pd.read_excel(file_path, sheet_name="Year 2009-2010", engine="openpyxl")
sheet_2010_2011 = pd.read_excel(file_path, sheet_name="Year 2010-2011", engine="openpyxl")
raw_pd = pd.concat([sheet_2009_2010, sheet_2010_2011], ignore_index=True)

raw_pd.columns = [re.sub(r'[ ,;{}()\n\t=]', '_', col) for col in raw_pd.columns]

object_cols = raw_pd.select_dtypes(include="object").columns
raw_pd[object_cols] = raw_pd[object_cols].astype("string")

In [0]:
from pyspark.sql.functions import lit, current_timestamp

bronze_df = (
    spark.createDataFrame(raw_pd)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("online_retail_II.xlsx"))
)

bronze_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_online_retail")

In [0]:
row_count = spark.table("workspace.default.bronze_online_retail").count()
assert row_count == 1067371, f"Unexpected row count: {row_count}"
print(f"Bronze validated: {row_count} rows")